# 02 — Gold status (v8)
Status and review profile for `gold_v2_eval_v8_public.jsonl` using central v8 loader/modules (`src.config.GOLD_PATH`, `src.eval.gold_rows`). Read-only; no gold/signature/split/freeze changes.

In [ ]:
# Parameters
# gold_path defaults to central v8 config (GOLD_PATH)


In [1]:
import os, json, hashlib
from pathlib import Path
from collections import Counter
_root = Path.cwd().resolve()
if not (_root / 'src').is_dir():
    _root = _root.parent
os.chdir(_root)
from src.config import GOLD_PATH
from src.eval.gold_rows import load_rows
gold = Path(GOLD_PATH)
rows = load_rows(str(gold), active_only=True)
if not rows:
    rows = [json.loads(l) for l in gold.read_text(encoding='utf-8').splitlines() if l.strip()]
print('gold_path:', gold)
print('dataset_sha256:', hashlib.sha256(gold.read_bytes()).hexdigest())
print('total_rows:', len(rows))
print('categories:', dict(Counter(r.get('category') for r in rows)))
print('review_status:', dict(Counter(r.get('review_status') for r in rows)))
print('status:', dict(Counter(r.get('status','active') for r in rows)))
print('split:', dict(Counter(r.get('split') for r in rows)))
print('expected_behaviour:', dict(Counter(r.get('expected_behaviour') for r in rows)))
print('doc_type:', dict(Counter(r.get('doc_type') for r in rows)))
ab = [r for r in rows if r.get('expected_behaviour')=='abstain']
print('abstention_subtype_all:', dict(Counter(r.get('abstention_subtype') for r in ab)))
print('abstention_subtype_by_split:', {s: dict(Counter(r.get('abstention_subtype') for r in ab if r.get('split')==s)) for s in ['dev','holdout']})
print('abstention_subtype_by_doc_type:', {d: dict(Counter(r.get('abstention_subtype') for r in ab if r.get('doc_type')==d)) for d in sorted({r.get('doc_type') for r in rows})})
two_reviewers = sum(len(r.get('reviewers') or []) >= 2 or (r.get('reviewer_count') or 0) >= 2 or r.get('peer_reviewed') for r in rows)
self_reviewed = sum(1 for r in rows if r.get('review_status')=='self_reviewed')
print('self_reviewed_rows:', self_reviewed)
print('rows_with_two_reviewers:', two_reviewers)
print('STATEMENT: all rows are self_reviewed =', self_reviewed == len(rows) and all(r.get('review_status')=='self_reviewed' for r in rows))
print('STATEMENT: zero rows have two reviewers =', two_reviewers == 0)


gold_path: data/eval/datasets/gold_v2_eval_v8_public.jsonl
dataset_sha256: 828b6704037297cd2e86da4c514edd9665b1945ee9c062ae7e5a15e417f0b09c
total_rows: 97
categories: {'cross_company_comparison': 5, 'single_fact_lookup': 26, 'numeric_from_table': 9, 'multi_hop': 12, 'unanswerable_abstain': 37, 'same_company_cross_period': 8}
review_status: {'self_reviewed': 97}
status: {'active': 97}
split: {'dev': 73, 'holdout': 24}
expected_behaviour: {'answer': 60, 'abstain': 37}
doc_type: {'annual': 77, 'quarterly': 20}
abstention_subtype_all: {'not_in_corpus': 31, 'near_miss_absent': 1, 'out_of_scope': 1, 'metric_not_applicable': 4}
abstention_subtype_by_split: {'dev': {'not_in_corpus': 23, 'near_miss_absent': 1, 'out_of_scope': 1, 'metric_not_applicable': 3}, 'holdout': {'not_in_corpus': 8, 'metric_not_applicable': 1}}
abstention_subtype_by_doc_type: {'annual': {'not_in_corpus': 31, 'near_miss_absent': 1, 'out_of_scope': 1}, 'quarterly': {'metric_not_applicable': 4}}
self_reviewed_rows: 97
rows_w

**Review/evidence limitations (transparent):**
- All 97 rows are `self_reviewed` and **zero** have two reviewers / peer review metadata indicating independent dual review; this notebook does **not** claim independent human verification.
- `reviewer_count`/`peer_reviewed` fields are reported as recorded in dataset; review quality/sign-off depth may vary.
- Evidence fields are present as recorded, but this status view does not re-verify evidence spans, page anchors, or answer correctness against source PDFs.
- Holdout split is reported for completeness only; evaluation runs must exclude holdout rows.